# 📄➡️📊 Conversor de PDF a Excel

Convierte PDFs a Excel (también CSV o JSON) usando el proyecto
[TsvetanG2/PDF-To-Excel-Converter](https://github.com/TsvetanG2/PDF-To-Excel-Converter) (licencia MIT).

## Cómo usarlo
1. **Menú Entorno de ejecución → Ejecutar todas** (o `Ctrl+F9`).
2. En el **Paso 2**, elegí el modo y el formato.
3. En el **Paso 3**, tocá **"Elegir archivos"** y subí uno o varios PDFs.
4. El archivo convertido se descarga solo (varios PDFs → un `.zip`).

> **Modos:**
> - **Todo el texto + tablas**: todo el contenido del PDF en una sola hoja.
> - **Solo tablas**: cada tabla detectada va en una hoja separada.
>
> **PDFs escaneados** (fotos o escaneos de papel): se detectan solos y se les reconoce
> el texto (OCR) en español e inglés antes de convertir. Tarda unos segundos más por página.
>
> Los archivos se procesan en una máquina temporal de Google que solo usa quien ejecuta el notebook.
> No quedan guardados en ningún lado al cerrar la sesión.

## Paso 1 · Instalar (solo la primera vez por sesión, 1-2 minutos)

In [ ]:
#@title Instalar el conversor
import os, shutil, subprocess

REPO_DIR = "/content/PDF-To-Excel-Converter"
if not os.path.exists(REPO_DIR):
    !git clone -q --depth 1 https://github.com/TsvetanG2/PDF-To-Excel-Converter.git {REPO_DIR}

# El modo "Solo tablas" usa tabula, que necesita Java
if shutil.which("java") is None:
    !apt-get -qq update && apt-get -qq install -y default-jre-headless > /dev/null


# OCR para PDFs escaneados (Tesseract con español + Ghostscript)
if shutil.which("tesseract") is None or shutil.which("gs") is None:
    !apt-get -qq update && apt-get -qq install -y tesseract-ocr tesseract-ocr-spa ghostscript > /dev/null

!pip install -q pdfplumber tabula-py openpyxl PyMuPDF pandas Flask Werkzeug ocrmypdf

print("✅ Listo. Seguí con el Paso 2.")

## Paso 2 · Elegir opciones

In [ ]:
#@title Opciones de conversión
modo = "Todo el texto + tablas"  #@param ["Todo el texto + tablas", "Solo tablas"]
formato = "xlsx"  #@param ["xlsx", "csv", "json"]
#@markdown **OCR (para PDFs escaneados):** "Automático" lo aplica solo a las páginas sin texto.
#@markdown Usá "Forzar" si el PDF tiene texto pero sale con caracteres raros.
ocr = "Automático"  #@param ["Automático", "Forzar en todas las páginas", "Desactivado"]

MODO = {"Todo el texto + tablas": "allText", "Solo tablas": "tablesOnly"}[modo]
OCR = {"Automático": "auto", "Forzar en todas las páginas": "force", "Desactivado": "off"}[ocr]
print(f"Modo: {modo} | Formato: {formato} | OCR: {ocr}")

## Paso 3 · Subir PDFs y convertir

In [ ]:
#@title Subir PDFs y descargar el resultado
import sys, os, json, zipfile, logging
from datetime import datetime
import pandas as pd

sys.path.insert(0, "/content/PDF-To-Excel-Converter/ConverterApp")
import pdftoexcel as conv
logging.getLogger("pdf_converter").setLevel(logging.WARNING)  # menos ruido en pantalla

def convertir(pdf_path, modo, formato, out_dir):
    base = os.path.splitext(os.path.basename(pdf_path))[0]
    out = os.path.join(out_dir, f"{base}.{formato}")
    if modo == "allText":
        texto, tablas = conv.extract_pdf_content(pdf_path)
        if formato == "xlsx":
            conv.create_excel(texto, tablas).save(out)
        elif formato == "csv":
            conv.create_csv(texto, tablas, out)
        else:
            conv.create_json(texto, tablas, out)
    else:
        tablas = conv.extract_tables_from_pdf(pdf_path)
        if not tablas:
            return None
        if formato == "xlsx":
            conv.write_tables_to_excel(tablas, out)
        elif formato == "csv":
            pd.concat([pd.DataFrame(t.values, columns=t.columns) for t in tablas],
                      ignore_index=True).to_csv(out, index=False)
        else:
            data = {"tables": [{"table_number": i + 1,
                                "columns": [str(c) for c in t.columns],
                                "data": t.astype(str).to_dict("records")}
                               for i, t in enumerate(tablas)]}
            with open(out, "w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False, indent=2)
    return out

import pdfplumber, ocrmypdf
logging.getLogger("ocrmypdf").setLevel(logging.ERROR)

def paginas_sin_texto(pdf_path):
    with pdfplumber.open(pdf_path) as pdf:
        return sum(1 for p in pdf.pages if len(p.chars) < 10)

def aplicar_ocr(pdf_path, modo_ocr):
    # Devuelve la ruta del PDF a convertir (con texto reconocido si hizo falta)
    if modo_ocr == "off":
        return pdf_path
    if modo_ocr == "auto":
        sin_texto = paginas_sin_texto(pdf_path)
        if sin_texto == 0:
            return pdf_path
        print(f"🔍 {os.path.basename(pdf_path)}: {sin_texto} página(s) escaneada(s), reconociendo texto...")
    else:
        print(f"🔍 {os.path.basename(pdf_path)}: reconociendo texto en todas las páginas...")
    salida = pdf_path[:-4] + "_ocr.pdf"
    try:
        ocrmypdf.ocr(pdf_path, salida, language="spa+eng", rotate_pages=True, deskew=True,
                     skip_text=(modo_ocr == "auto"), force_ocr=(modo_ocr == "force"),
                     progress_bar=False)
        return salida
    except Exception as e:
        print(f"   ⚠️ No se pudo aplicar OCR ({type(e).__name__}); se convierte sin OCR")
        return pdf_path

from google.colab import files

IN_DIR, OUT_DIR = "/content/entrada", "/content/salida"
for d in (IN_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)
    for f in os.listdir(d):
        os.remove(os.path.join(d, f))

subidos = files.upload()
resultados = []
for nombre, contenido in subidos.items():
    if not nombre.lower().endswith(".pdf"):
        print(f"⏭️  {nombre}: no es PDF, se omite")
        continue
    ruta = os.path.join(IN_DIR, nombre)
    with open(ruta, "wb") as f:
        f.write(contenido)
    try:
        ruta_final = aplicar_ocr(ruta, OCR)
        out = convertir(ruta_final, MODO, formato, OUT_DIR)
        if out and ruta_final != ruta:  # que el resultado conserve el nombre original
            nuevo = out.replace("_ocr.", ".")
            os.replace(out, nuevo)
            out = nuevo
        if out:
            resultados.append(out)
            print(f"✅ {nombre} → {os.path.basename(out)}")
        else:
            print(f"⚠️  {nombre}: no se encontraron tablas")
    except Exception as e:
        print(f"❌ {nombre}: error ({e})")

if len(resultados) == 1:
    files.download(resultados[0])
elif len(resultados) > 1:
    zip_path = f"/content/convertidos_{datetime.now():%Y%m%d_%H%M%S}.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for r in resultados:
            z.write(r, os.path.basename(r))
    files.download(zip_path)
else:
    print("No se generó ningún archivo.")

---
## (Opcional) Usar la interfaz web original

Si preferís la página con arrastrar y soltar del proyecto original, ejecutá esta celda
y abrí el enlace que aparece. Para detenerla, tocá el botón ⏹ de la celda.

In [ ]:
#@title Abrir la interfaz web
import threading, time
from google.colab import output
import pdftoexcel as conv

threading.Thread(target=lambda: conv.app.run(host="0.0.0.0", port=5000, threaded=True),
                 daemon=True).start()
time.sleep(2)
output.serve_kernel_port_as_window(5000, anchor_text="👉 Abrir el conversor PDF a Excel")
while True:
    time.sleep(60)